In [48]:
import pandas as pd
import os
import time
import requests
from datetime import datetime, timezone


COLUMNS_TO_DROP = [
        "ScoringPlay.ScoreID",
        "ScoringPlay.ScoringPlayID",
        "ScoringPlay.Season",
        "ScoringPlay.SeasonType",
        "ScoringPlay.GameKey",
        "ScoringPlay.Week",
        "ScoringPlay.AwayTeam",
        "ScoringPlay.HomeTeam",
        "ScoringPlay.Date",
        "ScoringPlay.Sequence",
        "ScoringPlay.Quarter",
        "ScoringPlay.AwayScore",
        "ScoringPlay.HomeScore",
        "ScoringPlay.ScoreID",
        "PlayStats"
]
def load_csv_from_link(link):
    resp = requests.get(link, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if isinstance(data, list):
        data = data[0]

    if isinstance(data, dict):
        df = pd.json_normalize(data)
        return df

def extract_plays_from_sportsdata_pbp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the SportsDataIO play-by-play dataframe where 'Plays' is a nested column
    and returns one row per play, with game metadata attached.
    """
    if df.empty:
        return pd.DataFrame()

    if "Plays" not in df.columns:
        raise ValueError("No 'Plays' column found.")

    all_play_dfs = []

    for _, row in df.iterrows():
        plays = row["Plays"]
        if not isinstance(plays, list) or len(plays) == 0:
            continue

        # Keep score/game-level metadata
        game_meta = {
            col: row[col]
            for col in df.columns
            if col.startswith("Score.") and col in ["Score.GameEndDateTime", "Score.HomeTeam", "Score.AwayTeam", "Score.HomeScore", "Score.AwayScore"]
        }

        plays_df = pd.json_normalize(plays)
    
        plays_df = plays_df.drop(columns=COLUMNS_TO_DROP, errors="ignore")
        for k, v in game_meta.items():
            plays_df[k] = v

        all_play_dfs.append(plays_df)

    if not all_play_dfs:
        return pd.DataFrame()

    out = pd.concat(all_play_dfs, ignore_index=True, sort=False)
    out = convert_est_columns_to_utc(out)
    return out
def convert_est_columns_to_utc(df: pd.DataFrame) -> pd.DataFrame:
    """
    SportsDataIO archive timestamps appear to be in US Eastern time.
    This converts all likely date/time columns to UTC.

    Uses America/New_York instead of fixed EST so daylight saving time is handled correctly.
    """
    out = df.copy()

    date_cols = [
        col for col in out.columns
        if any(x in col.lower() for x in ["date", "created", "updated", "playtime"])
    ]

    for col in date_cols:
        parsed = pd.to_datetime(out[col], errors="coerce")

        # Skip columns that clearly are not datetime-like
        if parsed.notna().sum() == 0:
            continue

        # If timezone-naive, assume America/New_York, then convert to UTC
        if parsed.dt.tz is None:
            out[col] = (
                parsed
                .dt.tz_localize("America/New_York", ambiguous="NaT", nonexistent="shift_forward")
                .dt.tz_convert("UTC")
            )
        else:
            out[col] = parsed.dt.tz_convert("UTC")

    return out

def extract_liveodds_from_sportsdata_pbp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the SportsDataIO live odds dataframe where 'LiveOdds' is a nested column
    and returns one row per odd update, with game metadata attached.
    """
    if df.empty:
        return pd.DataFrame()

    if "LiveOdds" not in df.columns:
        raise ValueError("No 'LiveOdds' column found.")

    all_odds_dfs = []

    for _, row in df.iterrows():
        odds = row["LiveOdds"]
        if not isinstance(odds, list) or len(odds) == 0:
            continue

        # Keep score/game-level metadata
        game_meta = {
            col: row[col]
            for col in df.columns
            if col.startswith("Score.")
        }

        odds_df = pd.json_normalize(odds)

        for k, v in game_meta.items():
            odds_df[k] = v

        all_odds_dfs.append(odds_df)

    if not all_odds_dfs:
        return pd.DataFrame()

    out = pd.concat(all_odds_dfs, ignore_index=True, sort=False)

    out = convert_est_columns_to_utc(out)


    return out

In [49]:
df = load_csv_from_link(link='http://archive.sportsdata.io/v3/nfl/pbp/json/playbyplay/19056/2025-09-08-03-02.json')
extracted_df = extract_plays_from_sportsdata_pbp(df)

gameid = 'KXNFLGAME-25SEP07MIAIND'
output_path = f"data/playbyplay/{gameid}.csv"

if not extracted_df.empty:
    write_header = not os.path.exists(output_path)
    extracted_df.to_csv(output_path, mode="a", header=write_header, index=False)

In [50]:
df = load_csv_from_link(link='http://archive.sportsdata.io/v3/nfl/odds/json/livegameoddslinemovement/19056/2025-09-08-02-59.json')

extracted_odds = extract_liveodds_from_sportsdata_pbp(df)

output_path = f"data/{gameid}.csv"

if not extracted_odds.empty:
    write_header = not os.path.exists(output_path)
    extracted_odds.to_csv(output_path, mode="a", header=write_header, index=False)
